# OpenInsight - Colab Ingestion (GPU-Accelerated)

**Data sources:** PubMed XMLs, StatPearls, MedQuAD  
**Embedder:** SentenceTransformers on GPU (local inference)  
**Vector DB:** Zilliz Cloud  
**Document store:** MongoDB Atlas

> **Add secrets first:** Colab sidebar -> Secrets -> add each key listed below

In [ ]:
# ── CELL 1: Install dependencies ──
# Fix NumPy ABI compatibility (Colab ships numpy 2.x, pymilvus needs 1.x)
%pip install -q 'numpy<2.0'
%pip install -q --force-reinstall --no-deps pandas

import subprocess
subprocess.run(["apt-get", "install", "-qq", "-y", "tesseract-ocr"], capture_output=True)

%pip install -q \
    fastapi uvicorn python-dotenv "pydantic>=2.0" pydantic-settings \
    pymongo motor pypdf pdfplumber beautifulsoup4 requests httpx lxml biopython \
    sentence-transformers transformers "torch>=2.0" \
    'pymilvus>=2.5,<2.6' langchain langchain-community openai redis \
    tqdm loguru tenacity python-slugify pytesseract Pillow scispacy \
    certifi cohere huggingface_hub typer

# Verify numpy/pandas/pymilvus compatibility
import numpy as np
print(f'numpy: {np.__version__}')
import pandas as pd
print(f'pandas: {pd.__version__}')
from pymilvus import MilvusClient
print('pymilvus: OK')

print('\nDependencies installed.')


In [ ]:
# ── CELL 2: Clone / update repo ──
import sys
from pathlib import Path

REPO_DIR = Path('/content/openinsight')

if REPO_DIR.exists():
    %cd /content/openinsight
    !git pull origin restruct -q
    print('Repo updated.')
else:
    !git clone -b restruct https://github.com/Adi103-ETAI/openinsight.git /content/openinsight -q
    print('Repo cloned.')

%cd /content/openinsight
sys.path.insert(0, '/content/openinsight')
print(f'Working dir: {Path.cwd()}')

In [ ]:
# ── CELL 3: Configuration ──
import os
from pathlib import Path
from google.colab import userdata

def _s(name, default=''):
    try:
        return userdata.get(name) or default
    except Exception:
        return default

ZILLIZ_URI         = _s('ZILLIZ_URI')
ZILLIZ_TOKEN       = _s('ZILLIZ_TOKEN')
MONGODB_URL        = _s('MONGODB_URL')
MONGODB_DB         = _s('MONGODB_DB', 'openinsight')
NVIDIA_NIM_API_KEY = _s('NVIDIA_NIM_API_KEY')
HF_API_TOKEN       = _s('HF_API_TOKEN')
COHERE_API_KEY     = _s('COHERE_API_KEY')
NCBI_API_KEY       = _s('NCBI_API_KEY')
NCBI_EMAIL         = _s('NCBI_EMAIL', 'sentarc.ai@gmail.com')

# Set environment variables for the pipeline
os.environ['MONGODB_URL']           = MONGODB_URL
os.environ['MONGODB_DB']            = MONGODB_DB
os.environ['VECTOR_URI']            = ZILLIZ_URI
os.environ['VECTOR_TOKEN']          = ZILLIZ_TOKEN
os.environ['MILVUS_CLOUD']          = 'true'
os.environ['VECTOR_COLLECTION_V2']  = 'openinsight_v2'
os.environ['NVIDIA_NIM_API_KEY']    = NVIDIA_NIM_API_KEY
os.environ['EMBED_PROVIDER']        = 'local'
os.environ['RERANK_PROVIDER']       = 'local'
os.environ['HF_API_TOKEN']          = HF_API_TOKEN
os.environ['HF_EMBED_MODEL']        = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['DENSE_MODEL_NAME']      = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['RERANKER_MODEL_NAME']   = 'BAAI/bge-reranker-v2-m3'
os.environ['COHERE_API_KEY']        = COHERE_API_KEY
os.environ['NCBI_API_KEY']          = NCBI_API_KEY
os.environ['NCBI_EMAIL']            = NCBI_EMAIL
os.environ['SSL_CERT_FILE']         = __import__('certifi').where()

PUBMED_DATA_DIR  = '/content/pubmed_xml'
MEDQUAD_DATA_DIR = '/content/medquad_data'
Path(PUBMED_DATA_DIR).mkdir(parents=True, exist_ok=True)

print('Config ready.')
print(f'   Embed mode  : local (SentenceTransformers on GPU)')
print(f'   PubMed dir  : {PUBMED_DATA_DIR}')

In [ ]:
# ── CELL 4: Connectivity Test ──
import pymongo
import certifi
from urllib.parse import quote_plus
from pymilvus import MilvusClient

print('--- Connectivity Test ---')
try:
    mc = pymongo.MongoClient(
        MONGODB_URL,
        tlsCAFile=certifi.where(),
        serverSelectionTimeoutMS=5000,
        authSource='admin'
    )
    mc.admin.command('ping')
    print('MongoDB: Connected!')
except Exception as e:
    print(f'MongoDB Error: {e}')

try:
    zc = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
    cols = zc.list_collections()
    print(f'Zilliz: Connected! Collections: {cols}')
except Exception as e:
    print(f'Zilliz Error: {e}')

In [ ]:
# ── CELL 5: Download PubMed + StatPearls XMLs ──
import os, time
from pathlib import Path
from Bio import Entrez

Entrez.email = os.environ['NCBI_EMAIL']
Entrez.api_key = os.environ.get('NCBI_API_KEY') or None
SLEEP = 0.11 if Entrez.api_key else 0.35

QUERIES = [
    'StatPearls[book] diabetes mellitus',
    'StatPearls[book] hypertension management',
    'StatPearls[book] sepsis management',
    'tuberculosis drug resistant India treatment 2023',
    'snakebite envenomation India management protocol'
]

out_dir = Path(PUBMED_DATA_DIR)
saved = 0

for query in QUERIES:
    safe = query[:55].replace(' ', '_').replace('[', '').replace(']', '').replace('/', '-')
    fpath = out_dir / f"{safe}.xml"
    try:
        with Entrez.esearch(db='pubmed', term=query, retmax=10) as h:
            pmids = Entrez.read(h)['IdList']
        if pmids:
            with Entrez.efetch(db='pubmed', id=','.join(pmids), rettype='xml', retmode='xml') as h:
                data = h.read()
            with open(fpath, 'wb') as f:
                f.write(data if isinstance(data, bytes) else data.encode())
            saved += 1
            print(f'  Saved: {fpath.name}')
        time.sleep(SLEEP)
    except Exception as e:
        print(f'  {query[:30]}: {e}')

print(f'\nDone. Total XML files ready: {len(list(out_dir.glob("*.xml"))}')

## 6. Wipe Old Data (First Run Only)

In [ ]:
# ── WIPE ALL DATA (Run this once before first ingestion) ──
from pymilvus import MilvusClient
from pymongo import MongoClient

zilliz = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz.has_collection('openinsight_v2'):
    zilliz.drop_collection('openinsight_v2')
    print('Zilliz: Dropped collection openinsight_v2')
else:
    print('Zilliz: Collection does not exist yet')

mongo = MongoClient(MONGODB_URL)
db = mongo[MONGODB_DB]
for col_name in ['documents_v2', 'chunks_v2', 'failed_documents',
                  'ingestion_checkpoints', 'ingestion_metrics']:
    result = db[col_name].delete_many({})
    print(f'MongoDB {col_name}: Deleted {result.deleted_count} docs')

print('\nAll data wiped! Ready for fresh ingestion.')

In [ ]:
# ── CELL 6: Ingest PubMed data ──
from src.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline()

summary = await pipeline.ingest_directory(
    directory=PUBMED_DATA_DIR,
    source='pubmed',
    recreate_index=True,
    batch_size=5,
    resume=True
)

print('\nINGESTION COMPLETE (PubMed)')
for key, value in summary.items():
    print(f'  {key}: {value}')

## 7. Download & Ingest MedQuAD

In [ ]:
# ── Download MedQuAD ──
from pathlib import Path

MEDQUAD_DIR = Path(MEDQUAD_DATA_DIR)
MEDQUAD_REPO = MEDQUAD_DIR / 'MedQuAD'

if MEDQUAD_REPO.exists():
    print(f'MedQuAD already cloned at {MEDQUAD_REPO}')
else:
    MEDQUAD_DIR.mkdir(parents=True, exist_ok=True)
    !git clone https://github.com/abachaa/MedQuAD.git {MEDQUAD_REPO}
    print('MedQuAD cloned.')

xml_files = list(MEDQUAD_REPO.rglob('*.xml'))
print(f'Total XML files: {len(xml_files)}')
for subdir in sorted(MEDQUAD_REPO.iterdir()):
    if subdir.is_dir():
        count = len(list(subdir.rglob('*.xml')))
        print(f'  {subdir.name}: {count} XML files')

In [ ]:
# ── Ingest MedQuAD ──
from src.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline()

summary = await pipeline.ingest_directory(
    directory=str(MEDQUAD_REPO),
    source='medquad',
    recreate_index=False,  # Append to existing collection
    batch_size=50,
    resume=True,
)

print('\nINGESTION COMPLETE (MedQuAD)')
for key, value in summary.items():
    print(f'  {key}: {value}')

In [ ]:
# ── Verify all data ──
from pymilvus import MilvusClient
from pymongo import MongoClient
import os

collection_name = os.environ.get('VECTOR_COLLECTION_V2', 'openinsight_v2')

zilliz_client = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz_client.has_collection(collection_name):
    stats = zilliz_client.get_collection_stats(collection_name)
    print(f'Zilliz row count: {stats.get("row_count", "unknown")}')

mongo_client = MongoClient(MONGODB_URL)
db = mongo_client[MONGODB_DB]
print(f'documents_v2: {db.documents_v2.count_documents({})}')
print(f'chunks_v2: {db.chunks_v2.count_documents({})}')
print(f'failed_documents: {db.failed_documents.count_documents({})}')

for src in db.documents_v2.distinct('source_type'):
    count = db.documents_v2.count_documents({'source_type': src})
    print(f'  {src}: {count} documents')